# Fine-tuning Qwen2.5-Coder-14B cu LoRA
### Diagram Generator — Architecture Diagram from Code/Text

**Inainte sa rulezi:**
1. Runtime → Change runtime type → **A100 GPU**
2. Incarca `train.jsonl` si `val.jsonl` in Google Drive
3. Ruleaza celulele in ordine

## 1. Instalare dependinte

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl datasets transformers accelerate peft bitsandbytes

## 2. Verificare GPU

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

## 3. Incarcare model + LoRA

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 4096  # PUML-urile mari necesita context lung
dtype = None           # Auto-detect: bfloat16 pe A100
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-14B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("Model incarcat.")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)
model.print_trainable_parameters()

## 4. Incarcare dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Modifica calea daca fisierele sunt in alt folder
TRAIN_PATH = "/content/drive/MyDrive/dataset/train.jsonl"
VAL_PATH   = "/content/drive/MyDrive/dataset/val.jsonl"

from datasets import load_dataset

train_dataset = load_dataset("json", data_files=TRAIN_PATH, split="train")
val_dataset   = load_dataset("json", data_files=VAL_PATH,   split="train")

print(f"Train: {len(train_dataset)} samples")
print(f"Val:   {len(val_dataset)} samples")
print("\nExemplu sample:")
print(train_dataset[0]["messages"][0]["content"][:200])

## 5. Formatare date (ChatML template)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def format_sample(sample):
    """Aplica chat template pe fiecare sample."""
    text = tokenizer.apply_chat_template(
        sample["messages"],
        tokenize = False,
        add_generation_prompt = False,
    )
    return {"text": text}

train_dataset = train_dataset.map(format_sample, batched=False)
val_dataset   = val_dataset.map(format_sample,   batched=False)

print("Sample formatat:")
print(train_dataset[0]["text"][:500])

## 6. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # effective batch = 8
        num_train_epochs = 3,
        warmup_ratio = 0.05,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        evaluation_strategy = "steps",
        eval_steps = 50,
        save_strategy = "steps",
        save_steps = 100,
        save_total_limit = 2,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        output_dir = "/content/drive/MyDrive/lora_output",
        report_to = "none",
    ),
)

print("Incepe training-ul...")
trainer_stats = trainer.train()
print(f"Training complet. Loss final: {trainer_stats.training_loss:.4f}")

## 7. Salvare adapter LoRA

In [ ]:
LORA_SAVE_PATH = "/content/drive/MyDrive/diagram_lora_adapter"

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f"Adapter salvat in: {LORA_SAVE_PATH}")

## 8. Export GGUF (pentru Ollama / llama.cpp)

In [ ]:
GGUF_SAVE_PATH = "/content/drive/MyDrive/diagram_gguf"

# Q4_K_M = cel mai bun raport calitate/marime pentru inferenta locala
model.save_pretrained_gguf(
    GGUF_SAVE_PATH,
    tokenizer,
    quantization_method = "q4_k_m"
)
print(f"GGUF salvat in: {GGUF_SAVE_PATH}")

## 9. Test rapid dupa training

In [ ]:
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = """You are an expert software architecture assistant specialized in generating Mermaid architecture diagrams from Java Spring Boot projects.

You accept three types of input:
- A plain-text description of the system's architecture and business domain
- A list of user stories describing the system's features and actors
- A PlantUML class diagram extracted via static analysis of the source code

Your output is ALWAYS a single valid Mermaid diagram in graph TB layout. Nothing else — no explanation, no markdown fences, no commentary.

Diagram rules:
- Group components into subgraphs. Use only the subgraphs that apply: Controllers, Services, Data_Access, Security, Infrastructure.
- Include only architecturally significant components: classes annotated as Controller, Service, Repository, Entity, and security-related filters or configurations.
- Exclude: DTOs, POJOs, value objects, exceptions, constants, test classes, and application boot/initializer classes.
- Draw dependency arrows between subgraphs only, not between individual classes.
- Omit any subgraph that would contain zero components after filtering."""

test_input = """A Spring Boot REST API that implements JWT-based authentication.
Users can register, log in to receive a token, and access protected endpoints.
The system uses password hashing, security filters for token validation,
and a relational database for user persistence."""

messages = [
    {"role": "system",    "content": SYSTEM_PROMPT},
    {"role": "user",      "content": test_input},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    temperature = 0.1,
    do_sample = True,
)

result = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== OUTPUT MODEL ===")
print(result)